# Fase 4: Features por repositorio

Partimos de `data/processed/eventos_silver.parquet` (capa Silver de la Fase 3)
y construimos una tabla con un renglon por repositorio (capa **Gold**), lista
para clustering en la Fase 5.

**Actualizacion (Fase 7):** esto lo habia corrido con las 12 horas
originales (77,696 repos activos). Ahora hay ~124 horas, incluyendo un
bloque de 4 dias seguidos (1-4 junio 2026) que agregamos justamente para
tener algo de señal temporal real. Aprovecho eso y agrego 2 features
nuevos que antes no tenian sentido: `n_horas_distintas` y
`n_dias_distintos` -- cuantas horas/dias DIFERENTES vimos actividad del
mismo repo. Con el muestreo disperso de antes esto casi siempre daba 1
(porque un repo casi nunca cae en dos horas distintas por azar), pero
ahora, gracias al bloque continuo, un repo activo en varios dias seguidos
si puede tener n_dias_distintos > 1 -- eso es una señal de sostenibilidad
que antes no podiamos medir.

Pasos:
1. Filtrar a repos con **al menos 3 eventos** en la muestra (mismo criterio
   que la vez pasada; reviso mas abajo si sigue siendo razonable con mas datos).
2. Features numericos por repo: cantidad de cada tipo de evento, total de
   eventos, numero de actores (usuarios) distintos, y los 2 features
   temporales nuevos.
3. Features de texto: juntamos los mensajes de commits + titulos de issues/PR
   de cada repo en un solo texto, y le aplicamos TF-IDF.
4. Guardamos la tabla final (numericos + vector TF-IDF) en `data/processed/`.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, countDistinct, when, concat_ws, collect_list, transform, substring

spark = (SparkSession.builder
         .master("local[*]")
         .appName("features-gharchive")
         .config("spark.driver.memory", "6g")  # subido de 4g a 6g, dataset ~10x mas grande
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

df_silver = spark.read.parquet("../data/processed/eventos_silver.parquet")
print("eventos en Silver:", df_silver.count())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/11 23:04:40 WARN Utils: Your hostname, katana, resolves to a loopback address: 127.0.1.1; using 192.168.18.59 instead (on interface enp2s0)
26/07/11 23:04:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/07/11 23:04:40 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


eventos en Silver: 12047086


## Paso 1: filtrar repos con al menos 3 eventos

In [2]:
MIN_EVENTOS = 3

conteo_repos = df_silver.groupBy(col("repo.name").alias("repo")).agg(count("*").alias("n_total"))
repos_activos = conteo_repos.filter(col("n_total") >= MIN_EVENTOS)

print("repos totales:", conteo_repos.count())
print(f"repos con >= {MIN_EVENTOS} eventos:", repos_activos.count())

# Nos quedamos solo con los eventos de esos repos (para no procesar de mas).
# Ojo: renombramos la columna del lado derecho antes del join y la borramos
# despues, porque si no quedan dos columnas llamadas "repo" (la struct
# original de df_silver y esta), y Spark no sabe cual usar mas adelante.
repos_activos_key = repos_activos.select(col("repo").alias("repo_key"))
df_activo = df_silver.join(
    repos_activos_key,
    df_silver["repo.name"] == repos_activos_key["repo_key"],
    "inner",
).drop("repo_key")
print("eventos de repos activos:", df_activo.count())

repos totales: 3419971


repos con >= 3 eventos: 847133


eventos de repos activos: 8843042


## Paso 2: features numericos por repo

Contamos, para cada repo: cuantos eventos de cada tipo tuvo, el total,
cuantos actores (usuarios) distintos participaron, y (nuevo en Fase 7)
en cuantas horas/dias distintos lo vimos activo.

In [3]:
def contar_tipo(tipo):
    """Cuenta cuantos eventos de un tipo especifico tiene cada fila (1 o 0),
    para poder sumarlos despues con groupBy."""
    return count(when(col("type") == tipo, True))

# hora_bucket = la hora exacta de origen (ej "2026-06-03T13"), dia = solo
# la fecha (ej "2026-06-03"). Sirven para ver en cuantos momentos DISTINTOS
# aparece el mismo repo -- con el bloque continuo de 4 dias esto ya no da
# siempre 1, como pasaba antes con el muestreo disperso solo.
df_activo = df_activo.withColumn("hora_bucket", substring(col("created_at"), 1, 13))
df_activo = df_activo.withColumn("dia", substring(col("created_at"), 1, 10))

features_numericos = df_activo.groupBy(col("repo.name").alias("repo")).agg(
    contar_tipo("PushEvent").alias("n_push"),
    contar_tipo("IssuesEvent").alias("n_issues"),
    contar_tipo("PullRequestEvent").alias("n_pr"),
    contar_tipo("WatchEvent").alias("n_watch"),
    contar_tipo("ForkEvent").alias("n_fork"),
    count("*").alias("n_total"),
    countDistinct("actor.login").alias("n_actores"),
    countDistinct("hora_bucket").alias("n_horas_distintas"),
    countDistinct("dia").alias("n_dias_distintos"),
)

features_numericos.show(5)

# cuantos repos tienen actividad en mas de 1 dia distinto? esa es la
# señal temporal nueva que antes no teniamos
multi_dia = features_numericos.filter(col("n_dias_distintos") > 1).count()
print("repos vistos en mas de 1 dia distinto:", multi_dia)

+--------------------+------+--------+----+-------+------+-------+---------+-----------------+----------------+
|                repo|n_push|n_issues|n_pr|n_watch|n_fork|n_total|n_actores|n_horas_distintas|n_dias_distintos|
+--------------------+------+--------+----+-------+------+-------+---------+-----------------+----------------+
|  tigu77/hotdeal-web|     6|       0|   0|      0|     0|      6|        1|                5|               5|
|UmmarFarooq14/Lin...|     5|       0|   0|      0|     0|      5|        1|                2|               2|
|eshanthakur02-a11...|     4|       0|   0|      0|     0|      4|        1|                2|               1|
| Aquatham/haibeifen1|    40|       0|   0|      0|     0|     40|        1|                2|               2|
|LitheFiG/ytMusic-RPC|     3|       0|   0|      0|     0|      3|        1|                3|               2|
+--------------------+------+--------+----+-------+------+-------+---------+-----------------+----------

repos vistos en mas de 1 dia distinto: 455907


## Paso 3: texto por repo (para TF-IDF)

Cada evento puede traer texto distinto segun su tipo:
- `PushEvent`: mensajes de los commits (`payload.commits[].message`)
- `IssuesEvent`: titulo del issue (`payload.issue.title`)
- `PullRequestEvent`: titulo del pull request (`payload.pull_request.title`)
- `WatchEvent` / `ForkEvent`: no traen texto

Armamos una sola columna de texto por evento, y luego la juntamos toda por
repo en un solo documento (recordar: mucho de esto viene nulo, ver Fase 4
intro — usamos lo que haya disponible).

In [4]:
# transform() aplica una funcion a cada elemento de un arreglo (aca, sacar
# el mensaje de cada commit del push)
mensajes_commits = concat_ws(" ", transform(col("payload.commits"), lambda c: c["message"]))

texto_evento = (
    when(col("type") == "PushEvent", mensajes_commits)
    .when(col("type") == "IssuesEvent", col("payload.issue.title"))
    .when(col("type") == "PullRequestEvent", col("payload.pull_request.title"))
    .otherwise(None)
)

df_texto_evento = df_activo.withColumn("texto_evento", texto_evento).filter(
    col("texto_evento").isNotNull() & (col("texto_evento") != "")
)

print("eventos con texto util:", df_texto_evento.count())

texto_por_repo = df_texto_evento.groupBy(col("repo.name").alias("repo")).agg(
    concat_ws(" ", collect_list("texto_evento")).alias("texto")
)

texto_por_repo.show(3, truncate=100)

eventos con texto util: 894627


+---------------------------+----------------------------------------------------------------------------------------------------+
|                       repo|                                                                                               texto|
+---------------------------+----------------------------------------------------------------------------------------------------+
|          0213digital/Kerya|                                                                                               1 1 1|
|02OthvanextlSEW/Trading-bot|Update LICENSE Update LICENSE Update LICENSE Update LICENSE Update LICENSE Update LICENSE Update ...|
|            0ceanSlim/grain|                  Memory leak: client relay-pool writeHandler goroutine leaks on upstream disconnect|
+---------------------------+----------------------------------------------------------------------------------------------------+
only showing top 3 rows


## Paso 4: juntar numericos + texto, y aplicar TF-IDF

Unimos las dos tablas (numericos y texto) por repo. Los repos sin texto
quedan con texto vacio (su vector TF-IDF va a salir en ceros, lo cual esta
bien: simplemente no aportan señal de texto).

Para el texto usamos `CountVectorizer` (en vez de `HashingTF`) porque
guarda el vocabulario real -> despues, en la Fase 6, podemos mirar que
palabras pesan mas en cada cluster, en vez de solo numeros sin significado.

In [5]:
# left join: si un repo no tiene fila en texto_por_repo, queda con texto = null
gold = features_numericos.join(texto_por_repo, on="repo", how="left")
gold = gold.fillna({"texto": ""})

print("repos en la tabla final:", gold.count())
gold.show(5, truncate=60)

repos en la tabla final: 847133


+----------------------------------------------+------+--------+----+-------+------+-------+---------+-----------------+----------------+-----------------------+
|                                          repo|n_push|n_issues|n_pr|n_watch|n_fork|n_total|n_actores|n_horas_distintas|n_dias_distintos|                  texto|
+----------------------------------------------+------+--------+----+-------+------+-------+---------+-----------------+----------------+-----------------------+
|                           Aquatham/haibeifen1|    40|       0|   0|      0|     0|     40|        1|                2|               2|                       |
|                 LandryStewart/Landry-workload|  2348|       0|   0|      0|     0|   2348|        1|               34|               6|                       |
|                          LitheFiG/ytMusic-RPC|     3|       0|   0|      0|     0|      3|        1|                3|               2|                       |
|eshanthakur02-a11y/flash-ba

In [6]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer, IDF

# 1) separar el texto en palabras (todo a minuscula, solo letras/numeros)
tokenizer = RegexTokenizer(inputCol="texto", outputCol="palabras", pattern="\\W+", toLowercase=True)

# 2) quitar palabras muy comunes que no aportan (the, and, is, etc. - los
# mensajes de commits/issues en GitHub son mayormente en ingles)
remover = StopWordsRemover(inputCol="palabras", outputCol="palabras_limpias")

# 3) contar palabras (vocabulario de las 300 palabras mas frecuentes que
# aparezcan en al menos 5 repos, para no quedarnos con palabras raras sueltas)
cv = CountVectorizer(inputCol="palabras_limpias", outputCol="tf", vocabSize=300, minDF=5)

# 4) TF-IDF: pesa mas las palabras frecuentes en un repo pero raras en general
idf = IDF(inputCol="tf", outputCol="tfidf")

pipeline_texto = Pipeline(stages=[tokenizer, remover, cv, idf])
modelo_texto = pipeline_texto.fit(gold)
gold_con_texto = modelo_texto.transform(gold)

vocabulario = modelo_texto.stages[2].vocabulary
print("tamano del vocabulario:", len(vocabulario))
print("ejemplo de palabras del vocabulario:", vocabulario[:20])

tamano del vocabulario: 300
ejemplo de palabras del vocabulario: ['update', 'com', 'add', 'github', 'fix', 'https', '0', '2025', '1', 'merge', 'pull', 'feat', 'code', '2', 'test', 'request', 'commit', 'md', '3', '08']


## Paso 5: guardar la tabla Gold y el modelo de texto

Guardamos:
- La tabla final (numericos + `tfidf`) en `data/processed/features_repos.parquet`.
- El pipeline de texto ya entrenado (tokenizer + stopwords + vocabulario +
  IDF) en `models/pipeline_texto/`, para poder reusar el mismo vocabulario
  en la Fase 6 al interpretar los clusters.

In [7]:
RUTA_FEATURES = "../data/processed/features_repos.parquet"
RUTA_MODELO_TEXTO = "../models/pipeline_texto"

columnas_finales = ["repo", "n_push", "n_issues", "n_pr", "n_watch", "n_fork",
                     "n_total", "n_actores", "n_horas_distintas", "n_dias_distintos", "tfidf"]

gold_con_texto.select(columnas_finales).write.mode("overwrite").parquet(RUTA_FEATURES)
modelo_texto.write().overwrite().save(RUTA_MODELO_TEXTO)

# Verificacion rapida
verificacion = spark.read.parquet(RUTA_FEATURES)
print("filas guardadas:", verificacion.count())
verificacion.printSchema()

filas guardadas: 847133
root
 |-- repo: string (nullable = true)
 |-- n_push: long (nullable = true)
 |-- n_issues: long (nullable = true)
 |-- n_pr: long (nullable = true)
 |-- n_watch: long (nullable = true)
 |-- n_fork: long (nullable = true)
 |-- n_total: long (nullable = true)
 |-- n_actores: long (nullable = true)
 |-- n_horas_distintas: long (nullable = true)
 |-- n_dias_distintos: long (nullable = true)
 |-- tfidf: vector (nullable = true)

